# MNIST MLP3 — baseline comparison

This notebook compares the three clean optimizer baselines:

1. **SGD + momentum**
2. **AdamW**
3. **SGD + momentum + Muon**

Run the three training notebooks first. They must all use the same shared output root. The default is `baseline/runs/`; set `RG_BASELINE_OUTPUT_ROOT` before starting every kernel to redirect the artifacts, for example to `/tmp/rg_optimizers_baselines`.

The analysis loads the saved histories and checkpoints; it does **not** retrain. It compares train/test cross-entropy, accuracy, `exp(cross_entropy)` as a derived perplexity-like display, generalization gaps, timing, parameter norm, layerwise WeightWatcher alpha, ERG gap, midpoint trace-log, and effective-rank diagnostics. With one seed per optimizer, the comparison is descriptive; error bars require replicated seeds.

In [ ]:
from pathlib import Path
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    baseline_candidate = candidate / "baseline"
    if (baseline_candidate / "rg_baselines").is_dir():
        ROOT = baseline_candidate
        break
    if (candidate / "rg_baselines").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from a clone of CalculatedContent/rg_optimizers")

override = os.environ.get("RG_BASELINE_OUTPUT_ROOT")
OUTPUT_ROOT = (
    Path(override).expanduser().resolve()
    if override
    else (ROOT / "runs").resolve()
)
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    "SGD + momentum": "sgd_momentum",
    "AdamW": "adamw",
    "SGD + momentum + Muon": "sgd_momentum_muon",
}
ORDER = list(RUNS)
COLORS = dict(zip(ORDER, plt.get_cmap("tab10").colors[: len(ORDER)]))

print("baseline root:", ROOT.resolve())
print("shared output root:", OUTPUT_ROOT)
print("comparison output:", COMPARISON_DIR)

In [ ]:
REQUIRED = (
    "performance_by_epoch.csv",
    "spectral_metrics_by_epoch_and_layer.csv",
    "config.json",
    "final_state.pt",
)

runs = {}
checkpoint_rows = []
config_rows = []
for label, slug in RUNS.items():
    run_dir = OUTPUT_ROOT / slug
    missing = [name for name in REQUIRED if not (run_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"{label} is incomplete in {run_dir}; missing {missing}. "
            "Run its training notebook first with the same output root."
        )

    performance = pd.read_csv(run_dir / "performance_by_epoch.csv")
    spectral = pd.read_csv(run_dir / "spectral_metrics_by_epoch_and_layer.csv")
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))

    performance = performance.copy()
    performance["baseline"] = label
    performance["run_slug"] = slug
    performance["train_perplexity"] = np.exp(performance["train_loss"])
    performance["test_perplexity"] = np.exp(performance["test_loss"])
    performance["accuracy_gap"] = performance["train_accuracy"] - performance["test_accuracy"]
    performance["loss_gap"] = performance["test_loss"] - performance["train_loss"]
    performance["cumulative_time_sec"] = performance["epoch_total_time_sec"].cumsum()

    spectral = spectral.copy()
    spectral["baseline"] = label
    spectral["run_slug"] = slug

    epochs = tuple(sorted(performance["epoch"].astype(int).unique()))
    final_epoch = max(epochs)
    expected = {f"epoch_{epoch:03d}.pt" for epoch in range(1, final_epoch + 1)}
    checkpoint_dir = run_dir / "checkpoints"
    present = {path.name for path in checkpoint_dir.glob("epoch_*.pt")} if checkpoint_dir.is_dir() else set()
    missing_checkpoints = sorted(expected - present)
    checkpoint_rows.append({
        "baseline": label,
        "run_dir": str(run_dir),
        "final_epoch": final_epoch,
        "expected_epoch_checkpoints": len(expected),
        "found_epoch_checkpoints": len(expected & present),
        "missing_epoch_checkpoints": ", ".join(missing_checkpoints),
        "final_state_present": (run_dir / "final_state.pt").is_file(),
    })
    config_rows.append({"baseline": label, **config})
    runs[label] = {"performance": performance, "spectral": spectral, "epochs": epochs}

epoch_grids = {label: run["epochs"] for label, run in runs.items()}
reference_epochs = next(iter(epoch_grids.values()))
if any(epochs != reference_epochs for epochs in epoch_grids.values()):
    raise RuntimeError(f"Epoch grids do not match: {epoch_grids}")

checkpoint_inventory = pd.DataFrame(checkpoint_rows).set_index("baseline").loc[ORDER]
config_table = pd.DataFrame(config_rows).set_index("baseline").loc[ORDER]
display(checkpoint_inventory)
display(config_table)

incomplete = checkpoint_inventory["found_epoch_checkpoints"] < checkpoint_inventory["expected_epoch_checkpoints"]
if incomplete.any():
    warnings.warn(
        "Some epoch checkpoints are missing. Rerun the affected training notebook(s); "
        "the modified notebooks save and verify one checkpoint per trained epoch."
    )

In [ ]:
performance = pd.concat([runs[label]["performance"] for label in ORDER], ignore_index=True)
spectral = pd.concat([runs[label]["spectral"] for label in ORDER], ignore_index=True)
valid_spectral = spectral.loc[spectral["status"].eq("ok")].copy()
if valid_spectral.empty:
    raise RuntimeError("No WeightWatcher rows with status='ok' were found")

summary_rows = []
layer_rows = []
for label in ORDER:
    perf = runs[label]["performance"].sort_values("epoch")
    final = perf.iloc[-1]
    best_acc = perf.loc[perf["test_accuracy"].idxmax()]
    best_loss = perf.loc[perf["test_loss"].idxmin()]
    final_layers = valid_spectral.loc[
        valid_spectral["baseline"].eq(label)
        & valid_spectral["epoch"].eq(int(final["epoch"]))
    ].copy()
    if final_layers.empty:
        raise RuntimeError(f"{label} has no valid final-epoch spectral rows")

    summary_rows.append({
        "baseline": label,
        "final_epoch": int(final["epoch"]),
        "train_loss": final["train_loss"],
        "test_loss": final["test_loss"],
        "train_accuracy": final["train_accuracy"],
        "test_accuracy": final["test_accuracy"],
        "train_perplexity": final["train_perplexity"],
        "test_perplexity": final["test_perplexity"],
        "accuracy_gap": final["accuracy_gap"],
        "loss_gap": final["loss_gap"],
        "parameter_l2_norm": final["parameter_l2_norm"],
        "cumulative_time_sec": final["cumulative_time_sec"],
        "best_test_accuracy": best_acc["test_accuracy"],
        "best_accuracy_epoch": int(best_acc["epoch"]),
        "minimum_test_loss": best_loss["test_loss"],
        "minimum_loss_epoch": int(best_loss["epoch"]),
        "final_alpha_mean": final_layers["alpha"].mean(),
        "final_alpha_min": final_layers["alpha"].min(),
        "final_alpha_max": final_layers["alpha"].max(),
        "final_ERG_gap_mean": final_layers["ERG_gap"].mean(),
        "final_trace_log_midpoint_mean": final_layers["trace_log_midpoint_per_eval"].mean(),
        "final_stable_rank_mean": final_layers["stable_rank"].mean(),
        "final_participation_ratio_mean": final_layers["participation_ratio"].mean(),
        "final_midpoint_energy_fraction_mean": final_layers["midpoint_energy_fraction"].mean(),
    })
    layer_rows.append(final_layers[[
        "baseline", "epoch", "layer", "alpha", "ERG_gap",
        "trace_log_midpoint_per_eval", "stable_rank", "participation_ratio",
        "midpoint_energy_fraction",
    ]])

final_summary = pd.DataFrame(summary_rows).set_index("baseline").loc[ORDER].reset_index()
final_layer_metrics = pd.concat(layer_rows, ignore_index=True)

display(final_summary)
display(final_layer_metrics.sort_values(["layer", "baseline"]))

In [ ]:
def finish(filename: str) -> None:
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()

for suffix, ylabel in [
    ("loss", "Cross-entropy loss"),
    ("accuracy", "Accuracy"),
    ("perplexity", "exp(cross-entropy)"),
]:
    plt.figure(figsize=(10, 6))
    for label in ORDER:
        data = performance.loc[performance["baseline"].eq(label)]
        plt.plot(data["epoch"], data[f"train_{suffix}"], color=COLORS[label], linewidth=2, label=f"{label} — train")
        plt.plot(data["epoch"], data[f"test_{suffix}"], color=COLORS[label], linestyle="--", linewidth=2, label=f"{label} — test")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"MNIST MLP3 baseline comparison — {ylabel.lower()}")
    plt.grid(alpha=0.25)
    plt.legend(ncol=2)
    finish(f"{suffix}_comparison.png")

for metric, ylabel, filename in [
    ("accuracy_gap", "Train accuracy − test accuracy", "accuracy_gap_comparison.png"),
    ("loss_gap", "Test loss − train loss", "loss_gap_comparison.png"),
    ("parameter_l2_norm", "Whole-model parameter L2 norm", "parameter_norm_comparison.png"),
    ("cumulative_time_sec", "Cumulative measured time (seconds)", "cumulative_time_comparison.png"),
]:
    plt.figure(figsize=(10, 6))
    for label in ORDER:
        data = performance.loc[performance["baseline"].eq(label)]
        plt.plot(data["epoch"], data[metric], color=COLORS[label], linewidth=2, label=label)
    if metric in {"accuracy_gap", "loss_gap"}:
        plt.axhline(0.0, color="black", linewidth=1, alpha=0.5)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"MNIST MLP3 baseline comparison — {ylabel.lower()}")
    plt.grid(alpha=0.25)
    plt.legend()
    finish(filename)

In [ ]:
spectral_epoch = (
    valid_spectral.groupby(["baseline", "epoch"], as_index=False)
    .agg(
        alpha_mean=("alpha", "mean"),
        ERG_gap_mean=("ERG_gap", "mean"),
        trace_log_midpoint_mean=("trace_log_midpoint_per_eval", "mean"),
        stable_rank_mean=("stable_rank", "mean"),
    )
)

for metric, ylabel, filename, reference in [
    ("alpha_mean", "Mean WeightWatcher alpha across layers", "mean_alpha_comparison.png", 2.0),
    ("ERG_gap_mean", "Mean ERG gap across layers", "mean_ERG_gap_comparison.png", None),
    ("trace_log_midpoint_mean", "Mean midpoint trace-log per retained eigenvalue", "mean_trace_log_midpoint_comparison.png", 0.0),
    ("stable_rank_mean", "Mean stable rank across layers", "mean_stable_rank_comparison.png", None),
]:
    plt.figure(figsize=(10, 6))
    for label in ORDER:
        data = spectral_epoch.loc[spectral_epoch["baseline"].eq(label)]
        plt.plot(data["epoch"], data[metric], color=COLORS[label], linewidth=2, label=label)
    if reference is not None:
        plt.axhline(reference, color="black", linestyle=":", linewidth=1.5, label=f"reference = {reference:g}")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"MNIST MLP3 baseline comparison — {ylabel.lower()}")
    plt.grid(alpha=0.25)
    plt.legend()
    finish(filename)

for metric, ylabel, prefix, reference in [
    ("alpha", "WeightWatcher alpha", "layer_alpha", 2.0),
    ("ERG_gap", "ERG gap", "layer_ERG_gap", None),
]:
    for layer in sorted(valid_spectral["layer"].astype(str).unique()):
        plt.figure(figsize=(10, 6))
        layer_data = valid_spectral.loc[valid_spectral["layer"].astype(str).eq(layer)]
        for label in ORDER:
            data = layer_data.loc[layer_data["baseline"].eq(label)]
            plt.plot(data["epoch"], data[metric], color=COLORS[label], linewidth=2, label=label)
        if reference is not None:
            plt.axhline(reference, color="black", linestyle=":", linewidth=1.5, label=f"reference = {reference:g}")
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(f"{layer}: {ylabel} by optimizer")
        plt.grid(alpha=0.25)
        plt.legend()
        finish(f"{prefix}_{layer}.png")

In [ ]:
performance.to_csv(COMPARISON_DIR / "all_performance_by_epoch.csv", index=False)
spectral.to_csv(COMPARISON_DIR / "all_spectral_metrics_by_epoch_and_layer.csv", index=False)
spectral_epoch.to_csv(COMPARISON_DIR / "spectral_epoch_summary.csv", index=False)
final_summary.to_csv(COMPARISON_DIR / "final_baseline_summary.csv", index=False)
final_layer_metrics.to_csv(COMPARISON_DIR / "final_layer_metrics.csv", index=False)
checkpoint_inventory.reset_index().to_csv(COMPARISON_DIR / "checkpoint_inventory.csv", index=False)
config_table.reset_index().to_csv(COMPARISON_DIR / "optimizer_configurations.csv", index=False)

winner = final_summary.sort_values(["test_accuracy", "test_loss"], ascending=[False, True]).iloc[0]
print(
    "Best final test accuracy (descriptive, one seed each): "
    f"{winner['baseline']} — {winner['test_accuracy']:.6f}"
)
print("saved comparison artifacts:", COMPARISON_DIR.resolve())